# Network 멀티 에이전트 — 학습 진단 ⇄ 문제 생성

02 와 같은 **Network** 패턴(두 에이전트가 서로 핸드오프)을 교육 도메인에 적용한다:
- **dashboard_agent**: 학생 진단 결과를 코드로 시각화(약점 차트) + 구조화 진단
- **problem_agent**: 약점 단원에 대해 위키 정보를 찾아 객관식 문제 생성

```
START → dashboard_agent ⇄ problem_agent → ('FINAL ANSWER' 시) END
```

02 와 구조는 같지만, **한 에이전트에 도구를 여러 개** 붙이는 법과 **LLM 을 품은 도구**(`return_diagnosis`)를 보여준다.

> `OPENAI_API_KEY` 필요. 위키 검색에 `wikipedia` 패키지 사용.

## 환경 변수 준비

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료")

## 공용 시스템 프롬프트
[basics] 협업 지침 — 끝났으면 `FINAL ANSWER`, 자기 범위 밖이면 다음 에이전트로 핸드오프.

In [ ]:
def make_system_prompt(suffix: str) -> str:
    return (
        f"""You are a helpful AI assistant collaborating with other specialized assistants to complete tasks step by step.

        Use the tools provided and make as much progress as possible. Do not hesitate to act within your scope.

        Two cases:
        1. If you fully completed your part AND no other assistant is needed, return your result prefixed with \"FINAL ANSWER\".
        2. If parts require actions you cannot perform, summarize what you've done and hand off to the next assistant.

        \n{suffix}"""
    )

## 1. problem_agent — 위키 기반 문제 생성
[basics] `WikipediaQueryRun` 도구 + `create_react_agent`. 고유명사(인물/국가/사건)일 때만 위키 검색하도록 프롬프트로 제약.

In [ ]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

wikipedia = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
llm = ChatOpenAI(model="gpt-4o")

problem_agent = create_react_agent(
    llm,
    tools=[wikipedia],
    prompt=make_system_prompt(
        "You can only generate problems. "
        "Retrieve from Wikipedia only when the query is a proper noun (person/country/historical event). "
        "Do not retrieve for general or abstract topics. "
        "Give a 5-option multiple choice question in Korean. "
        "Do not return the answer, just the question and options. "
        "You are working with a diagnosis colleague."
    ),
)

In [ ]:
# 단독 테스트
response = problem_agent.invoke(
    {"messages": [{"role": "user", "content": "세종대왕의 업적에 대한 문제를 만들어주세요."}]}
)
response["messages"][-1].pretty_print()

## 2. dashboard_agent — 학습 진단 시각화

도구 2개를 가진다:
- `python_exec_tool`: 진단 점수를 차트로 (약점=빨강, 그 외=파랑, 한글 폰트)
- `return_diagnosis`: 진단 결과를 구조화 — **도구 안에서 LLM 을 다시 호출**하는 형태

> ⚠️ `exec` 임의 코드 실행 — 학습용. 한글 차트는 `Malgun Gothic`(윈도우) 등 환경별 폰트 설정 필요.

In [ ]:
from typing import Annotated, Dict
from langchain_core.tools import tool

@tool
def python_exec_tool(
    code: Annotated[str,
        "Python code to generate a student level chart. "
        "Comments/labels/titles in Korean. "
        "Use soft red ('#FF9999') for weak areas (score < 70), soft blue ('#87CEFA') otherwise. "
        "Set a Korean-compatible font to avoid display errors."],
):
    """Use this to execute python code. Print values you want to see with print(...)."""
    try:
        result = exec(code)
    except BaseException as e:
        return f"Failed to execute. Error: {repr(e)}"
    return (
        f"Successfully executed:\n{code}\nStdout: {result}\n\n"
        "If you have completed all tasks, respond with FINAL ANSWER."
    )

@tool
def return_diagnosis(
    diagnosis: Annotated[str, "The diagnosis result to return to the user."]
) -> Dict[str, str]:
    """Return the student knowledge level diagnosis in a structured format."""
    # 도구 안에서 LLM 을 다시 호출해 진단 결과를 다듬는다
    prompt = (
        "You are a student diagnosis assistant. You will receive a diagnosis result "
        "and must return it in a structured format.\n\nDiagnosis Result:\n" + diagnosis
    )
    return llm.invoke(prompt).content

In [ ]:
dashboard_agent = create_react_agent(
    llm,
    tools=[python_exec_tool, return_diagnosis],
    prompt=make_system_prompt(
        "You can only return student level (charts or feedback text). "
        "Do not generate any problems. "
        "If needed, hand off to the dedicated problem generation agent."
    ),
)

## 3. 노드 — 핸드오프 로직
[basics] 02 와 동일한 패턴: 에이전트 호출 → `FINAL ANSWER` 면 END, 아니면 상대에게 `Command(goto=...)`.

In [ ]:
from typing import Literal
from langchain_core.messages import BaseMessage, HumanMessage
from langgraph.graph import MessagesState, END
from langgraph.types import Command

def get_next_node(last_message: BaseMessage, goto: str):
    if "FINAL ANSWER" in last_message.content:
        return END
    return goto

def dashboard_node(state: MessagesState) -> Command[Literal["problem_agent", END]]:
    result = dashboard_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "problem_agent")
    result["messages"][-1] = HumanMessage(
        content=result["messages"][-1].content, name="dashboard_agent"
    )
    return Command(update={"messages": result["messages"]}, goto=goto)

def problem_node(state: MessagesState) -> Command[Literal["dashboard_agent", END]]:
    result = problem_agent.invoke(state)
    goto = get_next_node(result["messages"][-1], "dashboard_agent")
    result["messages"][-1] = HumanMessage(
        content=result["messages"][-1].content, name="problem_agent"
    )
    return Command(update={"messages": result["messages"]}, goto=goto)

## 4. 그래프 조립

In [ ]:
from langgraph.graph import StateGraph, START

graph_builder = StateGraph(MessagesState)
graph_builder.add_node("dashboard_agent", dashboard_node)
graph_builder.add_node("problem_agent", problem_node)
graph_builder.add_edge(START, "dashboard_agent")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

## 테스트

학생 성취도를 주면 → dashboard 가 약점을 진단/시각화하고 → problem 이 약점 단원 문제를 생성한다.

In [ ]:
response = graph.invoke(
    {"messages": [("user", """
    [
        {"chapter": "분수와 소수", "score": 88},
        {"chapter": "일차방정식", "score": 70},
        {"chapter": "도형과 각", "score": 55},
        {"chapter": "통계", "score": 61}
    ]
    위 성취도 학생의 대시보드를 시각화하고, 약점 단원에 필요한 연습 문제를 생성해주세요.
    문제를 생성하면 끝내세요.
    """)]},
    {"recursion_limit": 150},
)
for msg in response["messages"]:
    msg.pretty_print()

## 정리

- 02 와 같은 **Network** 패턴을 교육 도메인에 적용 (dashboard ⇄ problem)
- **한 에이전트에 도구 여러 개** 부여 (`python_exec_tool` + `return_diagnosis`)
- **LLM 을 품은 도구**: `return_diagnosis` 가 도구 내부에서 다시 `llm.invoke` — 도구가 단순 함수 이상이 될 수 있다
- 핸드오프/종료 로직은 02 와 동일 (`Command(goto)`, `FINAL ANSWER`)

다음: 중앙 관리자가 작업을 배분하는 **Supervisor** 구조.